# Take-Home Task: Probability and Hypothesis Testing

## Instructions for Participants

This notebook contains **20 industry-oriented tasks** from probability,
random variables, probability distributions, CLT, and hypothesis testing.

The datasets are generated inside the notebook. You do not need to download any
external file.

For each task:

1. Read the business problem carefully.
2. Inspect the dataframe given for that task.
3. Fill the missing code wherever you see `# TODO`.
4. Print the final answer clearly.
5. Write a short business conclusion in the markdown cell.

Do not only calculate a number. Explain what that number means for the business.

## Submission Expectation

Submit the completed notebook with:

- all TODO code filled,
- outputs visible,
- short conclusions written below each task.

## Common Setup

Run this cell once before starting the tasks.

In [1]:
import math
import itertools
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import binom, norm

np.random.seed(42)
alpha = 0.05

def decision(p_value, alpha=0.05):
    if p_value < alpha:
        return "Reject H0"
    return "Fail to reject H0"

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 100)

## Dataset Generation

Run this cell to create all datasets used in the 20 tasks.

In [2]:
# Task 1: Product configuration dataset
colors = ["Black", "White", "Silver", "Red", "Blue"]
models = ["Basic", "Sport", "Luxury"]
fuel_types = ["Petrol", "Diesel", "EV"]
product_options_df = pd.DataFrame(
    list(itertools.product(colors, models, fuel_types)),
    columns=["car_color", "model_type", "fuel_type"]
)

# Task 2: Marketing channel overlap
channel_overlap_df = pd.DataFrame({
    "customer_group": ["Email only", "App only", "Both Email and App", "Neither"],
    "customer_count": [420, 310, 170, 100]
})

# Task 3: Customer churn conditional probability
churn_df = pd.DataFrame({
    "customer_id": [f"CUST-{i:04d}" for i in range(1, 501)],
    "plan_type": np.random.choice(["Basic", "Premium"], size=500, p=[0.65, 0.35])
})
churn_probability = np.where(churn_df["plan_type"] == "Basic", 0.24, 0.11)
churn_df["churned"] = np.random.binomial(1, churn_probability)

# Task 4: Fraud alert Bayes theorem dataset
fraud_screening_df = pd.DataFrame({
    "metric": ["Fraud prevalence", "Alert if fraud", "Alert if genuine"],
    "probability": [0.015, 0.92, 0.06]
})

# Task 5: Confusion matrix data
loan_model_df = pd.DataFrame({
    "actual_default": np.random.binomial(1, 0.18, 800)
})
loan_model_df["predicted_default"] = np.where(
    loan_model_df["actual_default"] == 1,
    np.random.binomial(1, 0.78, 800),
    np.random.binomial(1, 0.12, 800)
)

# Task 6: Insurance claim random variable
claim_payout_df = pd.DataFrame({
    "claim_payout_inr": [0, 5000, 15000, 50000, 100000],
    "probability": [0.70, 0.16, 0.08, 0.04, 0.02]
})

# Task 7: Daily ticket count probability distribution
ticket_count_df = pd.DataFrame({
    "tickets_per_day": [0, 1, 2, 3, 4, 5],
    "probability": [0.05, 0.14, 0.24, 0.28, 0.18, 0.11]
})

# Task 8: Delivery-slot uniform distribution
delivery_slot_df = pd.DataFrame({
    "slot_id": [f"SLOT-{i}" for i in range(1, 9)],
    "slot_start_hour": [8, 10, 12, 14, 16, 18, 20, 22]
})

# Task 9: Employee team selection
employee_pool_df = pd.DataFrame({
    "employee_id": [f"EMP-{i:03d}" for i in range(1, 19)],
    "department": np.random.choice(["Sales", "Analytics", "Operations"], size=18)
})

# Task 10 and 11: Conversion and SLA binomial scenarios
binomial_scenarios_df = pd.DataFrame({
    "scenario": ["Campaign conversion", "SLA breach"],
    "n_trials": [60, 25],
    "success_probability": [0.08, 0.10]
})

# Task 12 and 13: Delivery time normal distribution
delivery_time_df = pd.DataFrame({
    "order_id": [f"ORD-{i:04d}" for i in range(1, 301)],
    "delivery_minutes": np.random.normal(loc=42, scale=7, size=300).round(1)
})

# Task 14: CLT sampling dataset
invoice_df = pd.DataFrame({
    "invoice_id": [f"INV-{i:05d}" for i in range(1, 5001)],
    "basket_value_inr": np.random.gamma(shape=3.0, scale=450, size=5000).round(0)
})

# Task 15: One-sample t-test
call_center_df = pd.DataFrame({
    "call_id": [f"CALL-{i:04d}" for i in range(1, 81)],
    "handling_time_minutes": np.random.normal(loc=6.4, scale=1.3, size=80).round(2)
})

# Task 16: Independent t-test
store_format_df = pd.DataFrame({
    "invoice_id": [f"BILL-{i:04d}" for i in range(1, 121)],
    "store_format": ["Express"] * 60 + ["Flagship"] * 60,
    "basket_value_inr": np.concatenate([
        np.random.normal(740, 120, 60),
        np.random.normal(815, 135, 60)
    ]).round(0)
})

# Task 17: Paired t-test
training_df = pd.DataFrame({
    "employee_id": [f"EMP-{i:03d}" for i in range(1, 46)],
    "before_training_score": np.random.normal(68, 8, 45).round(1)
})
training_df["after_training_score"] = (
    training_df["before_training_score"] + np.random.normal(5.5, 4, 45)
).round(1)

# Task 18: Chi-square independence
conversion_df = pd.DataFrame({
    "visitor_id": [f"VIS-{i:04d}" for i in range(1, 701)],
    "traffic_source": np.random.choice(["Organic", "Paid", "Referral"], size=700, p=[0.45, 0.35, 0.20])
})
conversion_rates = conversion_df["traffic_source"].map({"Organic": 0.09, "Paid": 0.14, "Referral": 0.11})
conversion_df["converted"] = np.random.binomial(1, conversion_rates)

# Task 19: One-way ANOVA
fulfillment_df = pd.DataFrame({
    "shipment_id": [f"SHP-{i:04d}" for i in range(1, 151)],
    "warehouse": np.repeat(["North", "West", "South"], 50),
    "fulfillment_hours": np.concatenate([
        np.random.normal(18, 3, 50),
        np.random.normal(20, 3.5, 50),
        np.random.normal(23, 4, 50)
    ]).round(2)
})

# Task 20: Mann-Whitney U test for ordinal ratings
app_rating_df = pd.DataFrame({
    "review_id": [f"REV-{i:04d}" for i in range(1, 121)],
    "app_version": ["Old_UI"] * 60 + ["New_UI"] * 60,
    "rating": np.concatenate([
        np.random.choice([1, 2, 3, 4, 5], size=60, p=[0.08, 0.15, 0.34, 0.30, 0.13]),
        np.random.choice([1, 2, 3, 4, 5], size=60, p=[0.04, 0.10, 0.24, 0.36, 0.26])
    ])
})

print("All datasets are created successfully.")
print("Example dataset: delivery_time_df")
display(delivery_time_df.head())

All datasets are created successfully.
Example dataset: delivery_time_df


,order_id,delivery_minutes
0,ORD-0001,41.7
1,ORD-0002,41.1
2,ORD-0003,44.3
3,ORD-0004,52.0
4,ORD-0005,49.6


## Task 1: Counting Principle for Product Configurations

### Business Problem

An automobile company allows customers to configure a car by choosing color,
model type, and fuel type.

### Your Task

1. Print the first 10 rows of `product_options_df`.
2. Calculate the total number of possible configurations using:
   - dataframe row count,
   - counting principle formula.
3. Confirm whether both answers match.

In [ ]:
display(product_options_df.head(10))

# TODO: calculate total configurations from dataframe
total_from_df = None

# TODO: calculate total configurations using counting principle
total_from_formula = None

print("Total configurations from dataframe:", total_from_df)
print("Total configurations from formula:", total_from_formula)
print("Do both match?", total_from_df == total_from_formula)

### Write Your Business Conclusion

Explain why the counting principle is useful for product, pricing, or catalog
analytics.

## Task 2: Probability of Non-Mutually Exclusive Marketing Reach

### Business Problem

A marketing team wants to know the probability that a customer was reached by
Email or App notification. Some customers received both.

### Your Task

Use `channel_overlap_df` to calculate:

1. P(Email)
2. P(App)
3. P(Email and App)
4. P(Email or App)

In [ ]:
display(channel_overlap_df)

total_customers = channel_overlap_df["customer_count"].sum()

# TODO: calculate counts for email, app, both
email_count = None
app_count = None
both_count = None

# TODO: calculate probabilities
p_email = None
p_app = None
p_both = None
p_email_or_app = None

print("P(Email):", p_email)
print("P(App):", p_app)
print("P(Email and App):", p_both)
print("P(Email or App):", p_email_or_app)

### Write Your Business Conclusion

Explain why subtracting overlap is important in marketing reach calculations.

## Task 3: Conditional Probability of Churn by Plan Type

### Business Problem

A subscription company wants to know whether Basic customers churn more often
than Premium customers.

### Your Task

Using `churn_df`, calculate:

1. P(Churn)
2. P(Churn | Basic)
3. P(Churn | Premium)

In [ ]:
display(churn_df.head())

# TODO: calculate overall churn probability
p_churn = None

# TODO: calculate conditional churn probability by plan type
p_churn_basic = None
p_churn_premium = None

print("P(Churn):", p_churn)
print("P(Churn | Basic):", p_churn_basic)
print("P(Churn | Premium):", p_churn_premium)

### Write Your Business Conclusion

Which plan type appears riskier? What action could the retention team take?

## Task 4: Bayes Theorem for Fraud Alert Precision

### Business Problem

A fraud detection rule raises alerts. Fraud is rare, so the business wants to
know the probability that a transaction is truly fraud when the alert is raised.

### Given

`fraud_screening_df` contains:

- P(Fraud)
- P(Alert | Fraud)
- P(Alert | Genuine)

### Your Task

Calculate P(Fraud | Alert) using Bayes theorem.

In [ ]:
display(fraud_screening_df)

p_fraud = fraud_screening_df.loc[fraud_screening_df["metric"] == "Fraud prevalence", "probability"].iloc[0]
p_alert_given_fraud = fraud_screening_df.loc[fraud_screening_df["metric"] == "Alert if fraud", "probability"].iloc[0]
p_alert_given_genuine = fraud_screening_df.loc[fraud_screening_df["metric"] == "Alert if genuine", "probability"].iloc[0]

# TODO: calculate P(Genuine)
p_genuine = None

# TODO: calculate P(Alert)
p_alert = None

# TODO: calculate P(Fraud | Alert)
p_fraud_given_alert = None

print("P(Fraud | Alert):", p_fraud_given_alert)

### Write Your Business Conclusion

Explain why a high detection rate can still produce many false alerts when fraud
prevalence is low.

## Task 5: False Positives and False Negatives in Loan Default Model

### Business Problem

A bank uses a model to predict loan default. The risk team wants to inspect false
positives and false negatives.

### Your Task

Using `loan_model_df`, calculate:

1. Confusion matrix
2. False positive count
3. False negative count
4. False positive rate
5. False negative rate

In [ ]:
display(loan_model_df.head())

# TODO: create confusion matrix using pd.crosstab
confusion = None
display(confusion)

# TODO: calculate FP, FN, TN, TP
fp = None
fn = None
tn = None
tp = None

# TODO: calculate rates
false_positive_rate = None
false_negative_rate = None

print("False positives:", fp)
print("False negatives:", fn)
print("False positive rate:", false_positive_rate)
print("False negative rate:", false_negative_rate)

### Write Your Business Conclusion

Which error is more expensive for a bank: false positive or false negative?
Explain your reasoning.

## Task 6: Expected Value of Insurance Claim Payout

### Business Problem

An insurance company wants to estimate the expected payout per policy based on
historical claim probabilities.

### Your Task

Using `claim_payout_df`, calculate the expected claim payout.

In [ ]:
display(claim_payout_df)

# TODO: verify probabilities sum to 1
probability_sum = None

# TODO: calculate expected payout
expected_payout = None

print("Probability sum:", probability_sum)
print("Expected payout per policy:", expected_payout)

### Write Your Business Conclusion

How can expected value help pricing or reserving decisions?

## Task 7: Cumulative Probability Distribution of Daily Tickets

### Business Problem

A support manager wants to know the probability that a day will have at most 3
priority tickets.

### Your Task

Using `ticket_count_df`, calculate:

1. Cumulative probability column
2. P(tickets per day <= 3)

In [ ]:
display(ticket_count_df)

# TODO: create cumulative probability column
ticket_count_df["cumulative_probability"] = None

# TODO: calculate P(X <= 3)
p_at_most_3 = None

display(ticket_count_df)
print("P(tickets per day <= 3):", p_at_most_3)

### Write Your Business Conclusion

How can cumulative probability help staffing decisions?

## Task 8: Uniform Probability for Delivery Slots

### Business Problem

A delivery app randomly assigns one of 8 delivery slots when demand is balanced.

### Your Task

Using `delivery_slot_df`, calculate:

1. Probability of any one slot
2. Probability that the assigned slot starts before 2 PM

In [ ]:
display(delivery_slot_df)

# TODO: calculate probability of one slot
p_one_slot = None

# TODO: calculate probability slot starts before 14:00
p_before_2pm = None

print("P(any one slot):", p_one_slot)
print("P(slot starts before 2 PM):", p_before_2pm)

### Write Your Business Conclusion

When is uniform distribution a reasonable assumption in operations?

## Task 9: Combinations for Project Team Selection

### Business Problem

A manager wants to choose 4 employees from a pool of 18 for a special project.
Order does not matter.

### Your Task

Using `employee_pool_df`, calculate the number of possible teams of 4.

In [ ]:
display(employee_pool_df)

n = len(employee_pool_df)
r = 4

# TODO: calculate combinations using math.comb
possible_teams = None

print("Possible teams of 4:", possible_teams)

### Write Your Business Conclusion

Why does combination logic matter in team selection or campaign audience design?

## Task 10: Binomial Probability of Campaign Conversions

### Business Problem

A campaign has a historical conversion probability of 8%. For 60 contacted
customers, the marketing team wants the probability of exactly 7 conversions.

### Your Task

Use the binomial distribution to calculate P(X = 7).

In [ ]:
display(binomial_scenarios_df)

n = 60
p_success = 0.08
x = 7

# TODO: calculate binomial probability P(X = 7)
p_exact_7 = None

print("P(exactly 7 conversions):", p_exact_7)

### Write Your Business Conclusion

Is exactly 7 conversions a rare or reasonable outcome?

## Task 11: Cumulative Binomial Probability of SLA Breaches

### Business Problem

Historically, 10% of shipments breach SLA. For 25 shipments, operations wants
the probability that at most 2 shipments breach SLA.

### Your Task

Use cumulative binomial probability to calculate P(X <= 2).

In [ ]:
n = 25
p_breach = 0.10
x = 2

# TODO: calculate cumulative binomial probability P(X <= 2)
p_at_most_2_breaches = None

print("P(at most 2 SLA breaches):", p_at_most_2_breaches)

### Write Your Business Conclusion

How can this probability help the operations team set daily risk expectations?

## Task 12: Normal Distribution Probability for Delivery Time

### Business Problem

Delivery time is approximately normal with mean and standard deviation estimated
from historical orders. The business wants to know the probability that an order
takes more than 50 minutes.

### Your Task

Using `delivery_time_df`, calculate P(delivery time > 50).

In [ ]:
display(delivery_time_df.head())

mu = delivery_time_df["delivery_minutes"].mean()
sigma = delivery_time_df["delivery_minutes"].std(ddof=1)

# TODO: calculate P(X > 50) using normal CDF
p_more_than_50 = None

print("Mean delivery time:", mu)
print("Std delivery time:", sigma)
print("P(delivery time > 50):", p_more_than_50)

### Write Your Business Conclusion

What does this probability mean for SLA planning?

## Task 13: Z-Score for Delivery-Time Anomaly

### Business Problem

A customer reports that an order took 61 minutes. The operations analyst wants
to know how unusual this is compared with historical delivery times.

### Your Task

Calculate the z-score for a 61-minute delivery.

In [ ]:
order_time = 61
mu = delivery_time_df["delivery_minutes"].mean()
sigma = delivery_time_df["delivery_minutes"].std(ddof=1)

# TODO: calculate z-score
z_score = None

print("Z-score for 61-minute delivery:", z_score)

### Write Your Business Conclusion

Is this delivery time normal variation or an operational anomaly?

## Task 14: Central Limit Theorem with Basket Values

### Business Problem

Basket values are highly skewed. The analytics team wants to show that sample
means become more normal as sample size increases.

### Your Task

Using `invoice_df`:

1. Draw 1000 samples of size 40.
2. Store each sample mean.
3. Calculate the mean and standard deviation of the sample means.

In [ ]:
display(invoice_df.head())

sample_means = []

for i in range(1000):
    # TODO: take a random sample of size 40 from basket_value_inr
    sample = None
    
    # TODO: append sample mean to sample_means
    pass

sample_means = pd.Series(sample_means)

print("Population mean:", invoice_df["basket_value_inr"].mean())
print("Mean of sample means:", None)  # TODO
print("Std of sample means:", None)   # TODO

### Write Your Business Conclusion

Explain how CLT supports confidence intervals and hypothesis testing in business
analytics.

## Task 15: One-Sample t-Test for Call Handling Time

### Business Problem

A call center promises average handling time of 6 minutes. The manager wants to
test whether actual handling time is significantly different from 6 minutes.

### Hypotheses

H0: Mean handling time = 6 minutes

H1: Mean handling time != 6 minutes

In [ ]:
display(call_center_df.head())

# TODO: run one-sample t-test against 6 minutes
statistic = None
p_value = None

print("Test statistic:", statistic)
print("p-value:", p_value)
print("Decision:", decision(p_value) if p_value is not None else "TODO")

### Write Your Business Conclusion

Should the manager believe that average handling time differs from the promised
6 minutes?

## Task 16: Independent t-Test for Store Format Basket Value

### Business Problem

A retailer wants to compare average basket value between Express stores and
Flagship stores.

### Hypotheses

H0: Mean basket value is the same for both store formats.

H1: Mean basket value is different between store formats.

In [ ]:
display(store_format_df.head())

express = None  # TODO: filter Express basket_value_inr
flagship = None # TODO: filter Flagship basket_value_inr

# TODO: run independent t-test
statistic = None
p_value = None

print("Test statistic:", statistic)
print("p-value:", p_value)
print("Decision:", decision(p_value) if p_value is not None else "TODO")

### Write Your Business Conclusion

Does store format appear to affect basket value?

## Task 17: Paired t-Test for Training Impact

### Business Problem

The same employees were scored before and after training. HR wants to know if
the training significantly changed scores.

### Hypotheses

H0: Mean before-training score equals mean after-training score.

H1: Mean before-training score is different from mean after-training score.

In [ ]:
display(training_df.head())

# TODO: run paired t-test
statistic = None
p_value = None

print("Test statistic:", statistic)
print("p-value:", p_value)
print("Decision:", decision(p_value) if p_value is not None else "TODO")

### Write Your Business Conclusion

Did the training program make a statistically significant difference?

## Task 18: Chi-Square Test of Independence for Traffic Source and Conversion

### Business Problem

The growth team wants to know whether conversion is related to traffic source.

### Hypotheses

H0: Traffic source and conversion are independent.

H1: Traffic source and conversion are associated.

In [ ]:
display(conversion_df.head())

# TODO: create contingency table
contingency_table = None
display(contingency_table)

# TODO: run chi-square test
chi2_stat = None
p_value = None
dof = None
expected = None

print("Chi-square statistic:", chi2_stat)
print("p-value:", p_value)
print("Decision:", decision(p_value) if p_value is not None else "TODO")

### Write Your Business Conclusion

Should the growth team treat traffic source as related to conversion behavior?

## Task 19: One-Way ANOVA for Warehouse Fulfillment Time

### Business Problem

An operations team compares average fulfillment time across North, West, and
South warehouses.

### Hypotheses

H0: Mean fulfillment time is equal across all warehouses.

H1: At least one warehouse has a different mean fulfillment time.

In [ ]:
display(fulfillment_df.head())

north = None  # TODO: filter North fulfillment_hours
west = None   # TODO: filter West fulfillment_hours
south = None  # TODO: filter South fulfillment_hours

# TODO: run one-way ANOVA
f_statistic = None
p_value = None

print("F statistic:", f_statistic)
print("p-value:", p_value)
print("Decision:", decision(p_value) if p_value is not None else "TODO")

### Write Your Business Conclusion

Are all warehouses performing similarly, or does at least one require attention?

## Task 20: Mann-Whitney U Test for App Ratings

### Business Problem

A product team compares 1-5 star ratings between old UI and new UI. Ratings are
ordinal, so a non-parametric test is preferred.

### Hypotheses

H0: Rating distribution is the same for Old UI and New UI.

H1: Rating distribution is different for Old UI and New UI.

In [ ]:
display(app_rating_df.head())

old_ui = None  # TODO: filter Old_UI ratings
new_ui = None  # TODO: filter New_UI ratings

# TODO: run Mann-Whitney U test
u_statistic = None
p_value = None

print("U statistic:", u_statistic)
print("p-value:", p_value)
print("Decision:", decision(p_value) if p_value is not None else "TODO")

### Write Your Business Conclusion

Do the ratings suggest that the new UI changed customer experience?

# Final Reflection

Answer these after completing all tasks:

1. Which tasks used probability only?
2. Which tasks used probability distributions?
3. Which tasks used hypothesis testing?
4. Which task was most similar to a real business problem you have seen?
5. What is the most important difference between calculating a probability and
   running a hypothesis test?